# causal-learn API Tutorial

This notebook demonstrates the native API of causal-learn library for causal discovery and causal effect estimation.

- **Purpose**: Explore causal-learn's core functionality for identifying and estimating causal relationships
- **Reference**: See [API.md](./API.md) for detailed documentation
- **Library**: [causal-learn](https://causal-learn.readthedocs.io/)

## Overview

causal-learn provides algorithms for:
- Causal discovery (PC, GES, FCI algorithms)
- Causal effect estimation (SEM, regression adjustment)
- Visualization of causal graphs

This notebook demonstrates the basic usage of these components.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import logging
import sys
import os

# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('')))
sys.path.insert(0, project_root)

# causal-learn imports (uncomment when installed)
# from causallearn.search.ConstraintBased.PC import pc
# from causallearn.search.ScoreBased.GES import ges
# from causallearn.search.ConstraintBased.FCI import fci

# Project utilities
from utils.utils_data_io import load_labor_data, prepare_features_for_causal_discovery
from utils.utils_post_processing import (
    discover_causal_structure,
    estimate_causal_effects,
    visualize_causal_graph
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


## Load Sample Data

Load and prepare data for causal discovery demonstration.


In [ ]:
# Load data (placeholder - replace with actual data loading)
# data = load_labor_data('Data/all.data.combined.csv')

# For demonstration, create sample data
np.random.seed(42)
n_samples = 1000
sample_data = pd.DataFrame({
    'inflation_rate': np.random.normal(2.5, 1.0, n_samples),
    'unemployment_rate': np.random.normal(5.0, 1.5, n_samples),
    'wage_growth': np.random.normal(2.0, 0.8, n_samples),
    'employment_rate': np.random.normal(95.0, 2.0, n_samples),
    'gdp_growth': np.random.normal(2.5, 1.2, n_samples)
})

# Prepare features for causal discovery
variables = ['inflation_rate', 'unemployment_rate', 'wage_growth', 'employment_rate', 'gdp_growth']
causal_data = prepare_features_for_causal_discovery(sample_data, variables)

print(f"Data shape: {causal_data.shape}")
print(f"\nFirst few rows:")
causal_data.head()


## Causal Discovery: PC Algorithm

The PC algorithm is a constraint-based method for causal discovery.


In [ ]:
# Discover causal structure using PC algorithm
causal_graph, edges = discover_causal_structure(
    data=causal_data,
    algorithm='PC',
    alpha=0.05,
    variables=variables
)

print(f"Discovered {len(edges)} causal relationships")
print(f"\nEdges: {edges}")


## Visualize Causal Graph

Visualize the discovered causal structure as a Directed Acyclic Graph (DAG).


In [ ]:
# Visualize the causal graph
visualize_causal_graph(
    graph=causal_graph,
    output_path=None,  # Display in notebook
    title='Causal Structure: Economic Factors → Employment Outcomes'
)


## Causal Effect Estimation

Estimate causal effects using Structural Equation Modeling (SEM).


In [ ]:
# Estimate effect of inflation on wage growth
inflation_effect = estimate_causal_effects(
    data=causal_data,
    causal_graph=causal_graph,
    treatment='inflation_rate',
    outcome='wage_growth',
    method='SEM'
)

print(f"Causal Effect (Inflation → Wage Growth): {inflation_effect['coefficient']:.4f}")
print(f"95% Confidence Interval: [{inflation_effect['ci_lower']:.4f}, {inflation_effect['ci_upper']:.4f}]")
print(f"P-value: {inflation_effect['p_value']:.4f}")
print(f"Standard Error: {inflation_effect['se']:.4f}")


## Multiple Causal Effects

Estimate effects for multiple treatment-outcome pairs.


In [ ]:
# Estimate effect of unemployment on wage growth
unemployment_effect = estimate_causal_effects(
    data=causal_data,
    causal_graph=causal_graph,
    treatment='unemployment_rate',
    outcome='wage_growth',
    method='SEM'
)

# Estimate effect of GDP growth on employment rate
gdp_effect = estimate_causal_effects(
    data=causal_data,
    causal_graph=causal_graph,
    treatment='gdp_growth',
    outcome='employment_rate',
    method='SEM'
)

# Display results
results_df = pd.DataFrame({
    'Treatment': ['Inflation Rate', 'Unemployment Rate', 'GDP Growth'],
    'Outcome': ['Wage Growth', 'Wage Growth', 'Employment Rate'],
    'Effect': [
        inflation_effect['coefficient'],
        unemployment_effect['coefficient'],
        gdp_effect['coefficient']
    ],
    'CI Lower': [
        inflation_effect['ci_lower'],
        unemployment_effect['ci_lower'],
        gdp_effect['ci_lower']
    ],
    'CI Upper': [
        inflation_effect['ci_upper'],
        unemployment_effect['ci_upper'],
        gdp_effect['ci_upper']
    ]
})

results_df


## Summary

This notebook demonstrated:
1. Loading and preparing data for causal discovery
2. Using PC algorithm to discover causal structure
3. Visualizing causal graphs
4. Estimating causal effects using SEM

For more details, see [API.md](./API.md).
